In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("./archive/UrbanSound8K.csv")
df.head()

In [2]:
import librosa
import resampy
# First restart kernel 
print("Librosa version:", librosa.__version__)
print("Resampy version:", resampy.__version__)

In [3]:
import os

file_path = "./archive/UrbanSound8/fold5/6508-9-0-1.wav"
if not os.path.exists(file_path):
    print(f"File not found: {file_path}")
else:
    print("File exists.")

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import warnings
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Load metadata
metadata_path = "./archive/UrbanSound8K.csv"
metadata = pd.read_csv(metadata_path)

# Function to extract MFCC and SSRP features
def extract_features(file_path, n_mfcc=40, n_ssrp=40):
    try:
        # Load audio file using librosa
        y, sr = librosa.load(file_path, sr=None, res_type='kaiser_fast')
    except Exception as e:
        warnings.warn(f"Error loading {file_path}: {e}")
        return None  # Skip this file
    
    # Extract MFCC features
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc = np.mean(mfcc.T, axis=0)
    
    # Extract SSRP features (custom implementation)
    stft = np.abs(librosa.stft(y))
    ssrp = librosa.feature.melspectrogram(S=stft, sr=sr, n_mels=n_ssrp)
    ssrp = np.mean(ssrp.T, axis=0)
    
    # Combine MFCC and SSRP features
    features = np.hstack((mfcc, ssrp))
    return features

# Extract features and labels
features = []
labels = []
for index, row in metadata.iterrows():
    file_path = os.path.join("./archive", "fold" + str(row["fold"]), row["slice_file_name"])
    class_id = row["classID"]
    
    # Extract features
    feature = extract_features(file_path)
    if feature is not None:  # Skip files that couldn't be loaded
        features.append(feature)
        labels.append(class_id)

# Convert to numpy arrays
X = np.array(features)
y = to_categorical(labels)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Bidirectional, Conv1D, MaxPooling1D, Flatten, Dropout

# Reshape data for CNN input
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# Model architecture
model = Sequential()

# 1D CNN layers
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

# Bi-LSTM layers
model.add(Bidirectional(LSTM(128, return_sequences=True)))
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.3))

# Fully connected layers
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(10, activation='softmax'))  # 10 classes for UrbanSound8K

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

In [ ]:
# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

In [ ]:
# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

# Plot training history
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
# Save the trained model
model.save("urbansound8k_bi_lstm_cnn_mfcc_ssrp.h5")